In [2]:
from selenium import webdriver
import time
import requests #fetches data from webpage
from bs4 import BeautifulSoup #extracts data we need
import pandas as pd #to store data as csv or excel format
from datetime import date

WRITE_FILE_PATH = r'c:\Interest_rate_files'
CHROME_DRIVER_PATH = 'c:\\Interest_rate_files\\chromedriver-win64\\chromedriver.exe'
URL = "https://www.iob.in/Domestic_Rates" 

date1 = (date.today()).strftime('%d%m%Y') #Get today's date and convert it to to dd/mm/yy format
file_name= WRITE_FILE_PATH + '\Interest_rate_' + date1 + '.csv' #Set directory path to write file

# Set up the Chrome driver
driver = webdriver.Chrome(executable_path=CHROME_DRIVER_PATH)
driver.get(URL) # Fetch the HTML page
time.sleep(5)  ## Wait for 5 secs the JavaScript to execute (you can adjust the sleep time if needed)
html = driver.page_source # Get the rendered HTML
driver.quit() # Close the driver

soup=BeautifulSoup(html,'html.parser')
#print(soup.prettify())
table=soup.find("table")
#print(table)
#table=soup.find_all("table",{'class': 'rates-table-main'})
#if len(table)>1:
#   table=table[0]
rows=table.find_all("tr")

print("Today's date =",date1)
print("Interest rate file name = ",file_name)

list_of_interest_rates = []
tax_saver_row = []
row_count = 0
for i in rows[1:]:  #Skip heading
    data=i.find_all("td")
    row=['Indian Overseas Bank']
    row.extend([tr.text for tr in data[:1]])
    row.extend([tr.text for tr in data[2:]])
    row.extend(['',str(float(row[2]) + 0.5) + '%',''])
    row[2] = row[2] + '%'
    if row_count == 12:
        tax_saver_row = ['Indian Overseas Bank', 'Tax-saver Fixed Deposit', row[2], '', row[4], '']
        list_of_interest_rates.append(tax_saver_row)
    list_of_interest_rates.append(row)
    row_count += 1

pd.set_option('display.max_columns', None)    
int_rate_df=pd.DataFrame(list_of_interest_rates,columns=['bank_name','tenure','gen_rate','annualised_gen_rate',
                            'sr_rate','annualised_sr_rate'])
print(int_rate_df)
int_rate_df.to_csv(file_name, mode='a', header=False, index=False)
print("\nIndian Overseas Bank = Success. Number of rows added = ", len(list_of_interest_rates))

Today's date = 09122024
Interest rate file name =  c:\Interest_rate_files\Interest_rate_09122024.csv
               bank_name                                tenure gen_rate  \
0   Indian Overseas Bank                            7-14 Days*    4.00%   
1   Indian Overseas Bank                            15-29 Days    4.50%   
2   Indian Overseas Bank                            30-45 Days    4.50%   
3   Indian Overseas Bank                            46-60 Days    4.50%   
4   Indian Overseas Bank                            61-90 Days    4.25%   
5   Indian Overseas Bank                           91-120 Days    4.75%   
6   Indian Overseas Bank                          121-179 Days    4.25%   
7   Indian Overseas Bank                          180-269 Days    5.75%   
8   Indian Overseas Bank                  270 Days to < 1 Year    5.75%   
9   Indian Overseas Bank  1 Year to < 2 Years(Except 444 Days)    7.10%   
10  Indian Overseas Bank                              444 Days    7.30%   